# Module: Data Science for Asset Management (KX09)
# Group project: Trustwise — Charity Investment Copilot

This notebook is the **proof** behind our five-pager. It answers one question for a charity trustee:

> *"How much of our cash can we safely invest, in what, and how bad could a bad year be in pounds?"*

**How to run it:** on Google Colab, click **Runtime → Run all** (3–5 minutes). Nothing to upload: the cover photo is stored inside the notebook.
The notebook does everything in order: **Part A** analysis → **Part B** business plan → **Part C** builds the final PDF `Trustwise_Five_Pager.pdf` with all numbers and charts filled in automatically.

| Section | What it does | Course session |
|---|---|---|
| 0 | Imports and settings | 0–2 |
| 1 | Download and clean ETF prices | 2 |
| 2 | Exploratory Data Analysis (EDA): volatility, Sharpe ratio, beta, correlation | 2 |
| 3 | The example charity | – |
| 4 | Safe cash buffer and risk profile | 4 |
| 5 | Portfolio optimisation: efficient frontier, tangency portfolio, CAL | 2, 4 |
| 6 | GARCH(1,1): a bad year in £ | 3, 8 |
| 7 | Stress tests: 2008, 2020, 2022 | 8 |
| 8 | Out-of-sample backtest | 9 |
| 9 | K-means clustering of the Charity Commission register | 5–6 |
| 10 | Summary numbers | – |
| 11 | Business plan: 3-year model | – |
| 12 | Build the PDF | – |

In [ ]:
# Install the libraries Colab does not have by default
!pip install -q yfinance arch weasyprint

# 0. Import Libraries and Settings
All our **assumptions** are in this cell, so they are easy to see and change. Nothing is hard-coded later.

In [ ]:
#Imports
import os
import io
import zipfile
import numpy as np                 # linear algebra
import pandas as pd                # data processing
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf              # Yahoo Finance prices (Session 2)
from scipy.optimize import minimize
from arch import arch_model        # GARCH (Session 3)
from sklearn.cluster import KMeans # clustering (Session 6)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Ignore harmless warnings
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

RANDOM_SEED = 42  # same seed = same results every time
np.random.seed(RANDOM_SEED)
os.makedirs("figures", exist_ok=True)

# Our fund universe: 8 index ETFs listed in London, in GBP (justETF, 13 Nov 2026)
FUNDS = {
    "CSP1.L": "US equity (S&P 500)",       # IE00B5BMR087, TER 0.07%
    "SMEA.L": "Europe equity",             # IE00B4K48X80, TER 0.12%
    "SJPA.L": "Japan equity",              # IE00B4L5YX21, TER 0.12%
    "CPJ1.L": "Pacific ex-Japan equity",   # IE00B52MJY50, TER 0.20%
    "EMIM.L": "Emerging markets equity",   # IE00BKM4GZ66, TER 0.18%
    "SGLN.L": "Gold",                      # IE00B4ND3602, TER 0.12%
    "IGLT.L": "UK gilts",                  # IE00B1FZSB30, TER 0.07%
    "SLXX.L": "UK corporate bonds",        # IE00B00FV011, TER 0.20%
}
CASH_TICKER = "ERNS.L"   # GBP ultrashort bond ETF (IE00BCRY6441): our "cash" / risk-free asset
BENCHMARK   = "SWDA.L"   # MSCI World: 100% world equities, for comparison only
START_DATE  = "2012-01-01"

# Portfolio rules (Trustee Act 2000: trustees must diversify)
MAX_WEIGHT = 0.40   # no fund above 40%
GOLD_MAX   = 0.10   # gold at most 10%

# Risk aversion A for each profile, used in y* = (E[Rp] - Rf) / (A * variance)  (Session 4)
# We chose these values so that Careful / Balanced / Growth keep clearly different shares in cash
A_BY_PROFILE  = {"low": 30, "medium": 16, "high": 10}
BUFFER_MONTHS = {"low": 12, "medium": 6, "high": 3}   # months of spending kept in cash
BANK_RATE = 0.04   # Bank of England Bank Rate, 13 Nov 2026 (only used in the text)

# Chart colours: same as the PDF
NAVY, TEAL, ORANGE, GREY = "#14213D", "#167A68", "#D9682B", "#5B6476"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("Settings loaded.")

# 1. Download and clean the data (Session 2)
We download **adjusted close prices** (dividends included) from Yahoo Finance, as in Session 2.
Funds started on different dates, so we keep only the period where **all** funds have data.

In [ ]:
ALL = list(FUNDS) + [CASH_TICKER, BENCHMARK]
raw = yf.download(ALL, start=START_DATE, auto_adjust=True, progress=False)["Close"]
raw = raw[ALL]
print("Data source: Yahoo Finance, downloaded on", pd.Timestamp.today().date())
print("Shape of data", raw.shape)

# EDA step 1: missing values and first available date of each fund
print("\nMissing values per fund:")
print(raw.isnull().sum())
print("\nFirst date with data:")
print(raw.apply(lambda s: s.first_valid_index()))

**Data problem we found:** Yahoo Finance sometimes quotes London ETFs in **pence** on some days and in **pounds** on others.
This creates fake jumps of ×100 (or ÷100) in one day. The cell below finds these jumps and corrects them.

In [ ]:
def fix_pence_jumps(serie):
    '''
    serie : daily prices of one fund
    If the price jumps up ~100 times in one day, that day is in pence -> divide by 100.
    If it drops ~100 times, that day is in pounds after pence -> multiply by 100.
    '''
    values = serie.dropna().values.astype(float)
    n_fixes = 0
    for i in range(1, len(values)):
        ratio = values[i] / values[i - 1]
        if ratio > 30:
            values[i] = values[i] / 100
            n_fixes = n_fixes + 1
        elif ratio < 1/30:
            values[i] = values[i] * 100
            n_fixes = n_fixes + 1
    return pd.Series(values, index=serie.dropna().index), n_fixes

fixed = {}
for ticker in raw.columns:
    fixed[ticker], n_fixes = fix_pence_jumps(raw[ticker])
    if n_fixes > 0:
        print(ticker, ": fixed", n_fixes, "pence/pound jumps")
raw = pd.DataFrame(fixed)[ALL]

# Fill short gaps (bank holidays), then keep the common period
prices = raw.ffill(limit=5).dropna()

# Sanity check: no fund should move more than 25% in one day
big_moves = (prices.pct_change(1).abs() > 0.25).sum()
print("Days with a move above 25% (should be 0):", big_moves.sum())
print("Common period used:", prices.index[0].date(), "->", prices.index[-1].date(), "(", len(prices), "days )")

# Returns (Session 2): daily for GARCH, monthly for the portfolio
daily   = prices.pct_change(1).dropna()
monthly = prices.resample("ME").last().pct_change(1).dropna()
prices.describe()

# 2. Exploratory Data Analysis (Session 2)
For each fund: yearly return, yearly **volatility**, **Sharpe ratio**, **beta** against world equities, and **max drawdown** (worst fall from a peak).
We use **monthly** returns (×12 per year): London ETFs have small gaps on some days, and monthly data is less noisy.
The risk-free rate is the average return of the cash ETF over the same period.

In [ ]:
n_months = 12
rf_hist = monthly[CASH_TICKER].mean() * n_months     # historical cash return per year

def sharpe_ratio(serie, rf=rf_hist):
    '''
    serie : Series of monthly returns of the fund or portfolio
    rf: annual return on a risk-free investment (here: the cash ETF)
    '''
    rp = serie.mean() * n_months                        # mean annualised return
    volatility = serie.std() * np.sqrt(n_months)        # annual volatility
    return (rp - rf) / volatility

def max_drawdown(serie):
    wealth = (1 + serie).cumprod()                      # value of £1 invested
    return (wealth / wealth.cummax() - 1).min()         # worst fall from a previous peak

# Beta of each fund against world equities: cov(fund, world) / var(world)
cov_m = monthly.cov()
names = []
for t in ALL:
    if t in FUNDS:
        names.append(FUNDS[t])
    elif t == CASH_TICKER:
        names.append("Cash (ultrashort)")
    else:
        names.append("World equity (benchmark)")

stats = pd.DataFrame(index=ALL)
stats["name"] = names
stats["return %"] = monthly.mean() * n_months * 100
stats["volatility %"] = monthly.std() * np.sqrt(n_months) * 100
stats["Sharpe"] = [sharpe_ratio(monthly[t]) for t in ALL]
stats["beta vs world"] = cov_m[BENCHMARK] / cov_m.loc[BENCHMARK, BENCHMARK]
stats["max drawdown %"] = [max_drawdown(monthly[t]) * 100 for t in ALL]
print("Risk-free rate used (cash ETF, historical):", round(rf_hist * 100, 2), "% per year")
stats.round(2)

In [ ]:
# Correlation coefficients between the funds' monthly returns (Session 2)
corr = monthly[list(FUNDS) + [BENCHMARK]].corr()
labels = [c.replace(".L", "") for c in corr.columns]
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, xticklabels=labels, yticklabels=labels, annot=True, fmt=".2f",
            cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 8})
plt.title("How our funds move together (monthly correlation)")
plt.tight_layout()
plt.savefig("figures/fig0_correlation.png", dpi=200)
plt.show()

**What to read here:** equity funds move together (correlation around 0.6–0.9). Gold and gilts have a low correlation with equities, so they can reduce risk in a crisis. This is why a charity should not hold only one type of fund.

# 3. The example charity
A **hypothetical** mid-sized charity, used to show results in pounds. Change the numbers to test another charity.

In [ ]:
charity = {
    "annual_spending":  600_000,    # £ per year
    "total_reserves":   1_000_000,  # £
    "restricted_funds": 200_000,    # £, kept apart and matched to project dates
    "income_stability": "medium",   # low / medium / high  -> sets the cash buffer
    "risk_tolerance":   "medium",   # low / medium / high  -> from the trustee questionnaire
}
charity

# 4. Safe cash buffer and risk profile (Session 4 + our rule)
1. Keep some **months of spending in cash** (more months if income is unstable).
2. Only the rest of the **unrestricted** reserves can be invested.
3. **Risk capacity** = how many months of spending the reserves cover. **Risk tolerance** = what the trustees are comfortable with.
4. **Final profile = the lower of the two.** A charity is never pushed to take more risk than it can afford.

In [ ]:
LEVELS = ["low", "medium", "high"]

months_covered = charity["total_reserves"] / (charity["annual_spending"] / 12)

# Risk capacity: can the charity afford a bad year?
if months_covered < 9 or charity["income_stability"] == "low":
    capacity = "low"
elif months_covered < 24:
    capacity = "medium"
else:
    capacity = "high"

# Final profile = the lower of capacity and tolerance
tolerance = charity["risk_tolerance"]
final_profile = LEVELS[min(LEVELS.index(capacity), LEVELS.index(tolerance))]
A = A_BY_PROFILE[final_profile]

buffer_months = BUFFER_MONTHS[charity["income_stability"]]
safe_cash    = charity["annual_spending"] / 12 * buffer_months
unrestricted = charity["total_reserves"] - charity["restricted_funds"]
investable   = max(unrestricted - safe_cash, 0)

print("Months of spending covered by reserves:", round(months_covered, 1))
print(f"Safe cash buffer ({buffer_months} months): £{safe_cash:,.0f}")
print(f"Unrestricted reserves: £{unrestricted:,.0f}")
print(f"Amount that can be invested: £{investable:,.0f}")
print("Risk capacity:", capacity, "| Risk tolerance:", tolerance, "| FINAL PROFILE:", final_profile, "(A =", A, ")")

# 5. Portfolio optimisation using the Efficient Frontier (Sessions 2 and 4)
We follow the Session 2 method (`scipy.optimize.minimize` with SLSQP), with two extra rules for charities: **max 40% per fund** and **gold ≤ 10%**.
- **Tangency portfolio** = the mix with the maximum Sharpe ratio.
- **Capital Allocation Line (CAL)** = mixes of cash and the tangency portfolio.
- Share in the risky funds: **y\* = (E[Rp] − Rf) / (A · σp²)**, between 0 and 1. The rest goes to the cash fund.

In [ ]:
risky        = list(FUNDS)
returns      = monthly[risky]
n_assets     = returns.shape[1]
mean_returns = returns.mean()
sigma        = returns.cov()
rf           = rf_hist

def portfolio_return(weight):
    return np.sum(mean_returns * weight) * 12

def portfolio_volatility(weight):
    return np.sqrt(np.dot(weight.T, np.dot(sigma, weight))) * np.sqrt(12)

def portfolio_performance(weight):
    return portfolio_return(weight), portfolio_volatility(weight)

# Constraints: weights sum to 1, gold <= 10%. Bounds: each fund between 0% and 40%
gold_i      = risky.index("SGLN.L")
constraints = [{'type': 'eq',   'fun': lambda x: np.sum(x) - 1},
               {'type': 'ineq', 'fun': lambda x: GOLD_MAX - x[gold_i]}]
bounds      = ((0, MAX_WEIGHT),) * n_assets
w0          = np.ones(n_assets) / n_assets   # start from equal weights (class uses a random start; this one is reproducible)

def negativeSR(weight):
    return_p, vol_p = portfolio_performance(weight)
    return -(return_p - rf) / vol_p

def max_sharpe_ratio():
    return minimize(negativeSR, w0, method='SLSQP', bounds=bounds, constraints=constraints)

def min_vol():
    return minimize(lambda x: portfolio_volatility(x)**2, w0, method='SLSQP', bounds=bounds, constraints=constraints)

def efficient_portfolio_target(target):
    cons = constraints + [{'type': 'eq', 'fun': lambda x: portfolio_return(x) - target}]
    return minimize(lambda x: portfolio_volatility(x)**2, w0, method='SLSQP', bounds=bounds, constraints=cons)

w_tan    = max_sharpe_ratio()['x']     # tangency portfolio
w_minvar = min_vol()['x']              # minimum volatility portfolio
er_t, sd_t = portfolio_performance(w_tan)

# Efficient frontier: for each target return, find the lowest-risk mix
frontier = []
for target in np.linspace(portfolio_return(w_minvar), mean_returns.max() * 12 * 0.98, 40):
    res = efficient_portfolio_target(target)
    if res.success and abs(portfolio_return(res.x) - target) < 1e-4:   # keep only targets the solver could reach
        frontier.append(portfolio_performance(res.x))
frontier = np.array(sorted(frontier, key=lambda p: p[1]))

def y_star(A_):
    '''Share invested in the tangency portfolio for risk aversion A_ (Session 4), between 0 and 1.'''
    y = (er_t - rf) / (A_ * sd_t**2)
    return float(min(max(y, 0), 1))

print("Tangency portfolio: return", round(er_t*100, 2), "% | volatility", round(sd_t*100, 2), "% | Sharpe", round((er_t-rf)/sd_t, 2))
for p in LEVELS:
    print(f"  {p:>6} profile (A={A_BY_PROFILE[p]}): y* = {y_star(A_BY_PROFILE[p]):.0%} in risky funds")

In [ ]:
# Portfolio simulation (as in Session 2): 10,000 random mixes, coloured by Sharpe ratio (only those within our limits are shown)
n_sim = 10000
rng = np.random.default_rng(RANDOM_SEED)            # own random generator: does not change the rest of the notebook
sim_weights = rng.dirichlet(np.ones(n_assets), n_sim)
ok = (sim_weights.max(axis=1) <= MAX_WEIGHT) & (sim_weights[:, gold_i] <= GOLD_MAX)   # keep only mixes that respect our limits
sim_weights = sim_weights[ok]
sim_ret = np.array([portfolio_return(w) for w in sim_weights])
sim_vol = np.array([portfolio_volatility(w) for w in sim_weights])

plt.figure(figsize=(9, 5.6))
plt.scatter(sim_vol*100, sim_ret*100, c=(sim_ret - rf)/sim_vol, cmap="Greys", s=3, alpha=0.4)
plt.plot(frontier[:, 1]*100, frontier[:, 0]*100, color=NAVY, lw=2.5, label="Efficient frontier (with our limits)")
for i, t in enumerate(risky):
    plt.scatter(np.sqrt(sigma.iloc[i, i]*12)*100, mean_returns.iloc[i]*12*100, color=GREY, s=30)
    plt.annotate(t.replace(".L", ""), (np.sqrt(sigma.iloc[i, i]*12)*100, mean_returns.iloc[i]*12*100), fontsize=8, color=GREY)
x = np.linspace(0, frontier[:, 1].max()*100, 50)
plt.plot(x, rf*100 + (er_t - rf)/sd_t * x, color=TEAL, ls="--", lw=2, label="Capital Allocation Line")
plt.scatter(sd_t*100, er_t*100, color=TEAL, marker="*", s=300, zorder=5, label="Tangency portfolio (max Sharpe)")
plt.scatter(0, rf*100, color=NAVY, s=60, marker="s", zorder=5, label="Cash")
for p, mk in zip(LEVELS, ["v", "o", "^"]):
    y = y_star(A_BY_PROFILE[p])
    plt.scatter(y*sd_t*100, (rf + y*(er_t-rf))*100, color=ORANGE, s=110, marker=mk, zorder=6, label=f"Charity, {p} risk (y*={y:.0%})")
plt.xlabel("Annualised volatility (%)")
plt.ylabel("Annualised return (%)")
plt.title("Which mix of funds gives the best return for each level of risk?")
plt.legend(fontsize=8, loc="lower right")
plt.xlim(left=0)
plt.tight_layout()
plt.savefig("figures/fig1_frontier.png", dpi=200)
plt.show()

In [ ]:
# Weights of each profile: y* in the tangency portfolio, 1 - y* in cash
rows = {}
for p in LEVELS:
    y = y_star(A_BY_PROFILE[p])
    rows[f"{p} risk (%)"] = list(np.round(w_tan * y * 100, 1)) + [round((1 - y) * 100, 1)]
weights = pd.DataFrame(rows, index=[FUNDS[t] for t in risky] + ["Cash (ultrashort)"])
weights[f"£ for example charity ({final_profile})"] = (weights[f"{final_profile} risk (%)"] / 100 * investable).round(-2)
weights

**What to read here:** the tangency mix is the same for everyone. What changes with the charity's profile is **how much** goes into it and how much stays in cash. This is the "two-fund separation" idea from Session 4, and it is easy to explain to a board.

**Why so much Japan and no UK shares?** The optimiser only looks at 2014–2026 risk and return, and our list has no UK equity fund. The weights show the method, not a recommendation.

# 6. GARCH(1,1): how bad could a bad year be, in pounds? (Sessions 3 and 8)
Volatility changes over time (calm and stormy periods). **GARCH(1,1)** models this and forecasts the risk for the **next 12 months**, not only the long-run average.
Then we compute a **1-in-20 bad year** (95% Value at Risk) for the money the charity invests.

**Prudent choice:** in a bad year we do **not** count on the expected return, because it comes from a very strong market (2014–2026). So: bad year = −1.645 × volatility.

In [ ]:
y_ch = y_star(A)
w_full = np.append(w_tan * y_ch, 1 - y_ch)                 # risky funds + cash (example charity)
port_daily  = daily[risky + [CASH_TICKER]] @ w_full        # daily returns of the charity's portfolio
risky_daily = daily[risky] @ w_tan                         # daily returns of the tangency (risky) part

# GARCH(1,1) on the risky part (returns in %). Cash has almost no volatility,
# so the risk of a profile = y* x volatility of the risky part.
am  = arch_model(risky_daily * 100, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
fit = am.fit(disp="off")
print(fit.summary())
alpha = fit.params["alpha[1]"]
beta  = fit.params["beta[1]"]
print("\nalpha + beta =", round(alpha + beta, 3), "(below 1 = shocks fade over time)")

# Forecast the variance for the next 252 trading days and add it up -> 12-month volatility
fc = fit.forecast(horizon=252)
vol_risky = np.sqrt(fc.variance.iloc[-1].sum()) / 100      # risky part, next 12 months
vol_1y = y_ch * vol_risky                                  # example charity
exp_1y = rf + y_ch * (er_t - rf)                           # expected return (section 5)

var95  = 1.645 * vol_1y                                    # 1-in-20 loss (95% VaR, zero mean)
cvar95 = vol_1y * np.exp(-1.645**2 / 2) / (np.sqrt(2 * np.pi) * 0.05)   # average loss in the worst 5% of years
bad_ret = -var95                                           # return in a 1-in-20 bad year

# Check with history: 5th percentile of all rolling 12-month returns
roll_1y = (1 + port_daily).rolling(252).apply(np.prod, raw=True) - 1
hist_5  = roll_1y.quantile(0.05)

print(f"Historical volatility of the risky part: {sd_t:.1%} | GARCH forecast for the next 12 months: {vol_risky:.1%}")
print(f"Example charity: expected return {exp_1y:.2%} | GARCH volatility {vol_1y:.2%}")
print(f"Bad year (1 in 20, VaR 95%): {bad_ret:+.1%}  ->  £{bad_ret*investable:+,.0f}")
print(f"Average of the worst 5% of years (CVaR 95%): {-cvar95:+.1%}  ->  £{-cvar95*investable:+,.0f}")
print(f"Check with history (5th percentile of rolling 12-month returns): {hist_5:+.1%}")

In [ ]:
start  = investable
values = [start, start * (1 + exp_1y), start * (1 - var95)]
labels = ["Invested today", "Normal year\n(expected)", "Bad year\n(1 in 20)"]

plt.figure(figsize=(8, 4.8))
bars = plt.bar(labels, values, color=[NAVY, TEAL, ORANGE], width=0.55)
for b, v in zip(bars, values):
    plt.text(b.get_x() + b.get_width()/2, v, f"£{v:,.0f}", ha="center", va="bottom", fontweight="bold")
plt.ylim(0, max(values) * 1.18)
plt.ylabel("Value after one year (£)")
plt.title(f"A bad year in pounds: £{start:,.0f} invested ({final_profile} risk profile)")
plt.tight_layout()
plt.savefig("figures/fig2_garch_bad_year.png", dpi=200)
plt.show()

# 7. Stress tests: 2008, 2020, 2022 (Session 8)
Most of our London ETFs did not exist in 2008. So we use **long-history ETFs that track the same indices** (mostly US-listed, in USD) as proxies.
This is a common approach; the currency effect is a limitation, and we say so in the PDF.
Each window goes from the market peak to the bottom.

In [ ]:
PROXY = {"CSP1.L": "SPY", "SMEA.L": "IEV", "SJPA.L": "EWJ", "CPJ1.L": "EPP", "EMIM.L": "EEM",
         "SGLN.L": "GLD", "IGLT.L": "IGLT.L", "SLXX.L": "SLXX.L", CASH_TICKER: "BIL", BENCHMARK: "ACWI"}
WINDOWS = {"2008 financial crisis":  ("2008-09-01", "2009-03-09"),
           "2020 Covid crash":       ("2020-02-19", "2020-03-23"),
           "2022 rates & inflation": ("2022-01-03", "2022-10-12")}

px = yf.download(sorted(set(PROXY.values())), start="2008-06-01", end="2022-12-31",
                 auto_adjust=True, progress=False)["Close"].ffill()

def window_return(ticker, start, end):
    '''Return of the proxy of `ticker` between two dates (nan if no data).'''
    s = px[PROXY[ticker]].loc[start:end].dropna()
    if len(s) > 2:
        return s.iloc[-1] / s.iloc[0] - 1
    return np.nan

stress = {}
for name, (start, end) in WINDOWS.items():
    r = {t: window_return(t, start, end) for t in PROXY}
    if np.isnan(r[CASH_TICKER]):
        r[CASH_TICKER] = 0
    ours = 0
    for w, t in zip(w_full, risky + [CASH_TICKER]):
        ours = ours + w * r[t]
    stress[name] = {"Our portfolio (%)": ours * 100, "100% world equities (%)": r[BENCHMARK] * 100,
                    "Cash (%)": r[CASH_TICKER] * 100, "Our change in £": ours * investable}
stress = pd.DataFrame(stress).T
stress.round(1)

# 8. Out-of-sample backtest (Session 9)
To avoid **overfitting**, the weights are estimated only with data available **at that time**:
- at the start of each year from 2020, we estimate the tangency portfolio with past data only (expanding window);
- we hold it for 12 months (charities review their investments once a year, CC14);
- we compare it with **cash** and **100% world equities**.

In [ ]:
def tangency_weights(train):
    '''Tangency portfolio + cash, estimated only on the training data.'''
    mu_  = train[risky].mean().values * 12
    cov_ = train[risky].cov().values * 12
    rf_  = train[CASH_TICKER].mean() * 12
    def neg_sharpe(w):
        return -(w @ mu_ - rf_) / np.sqrt(w @ cov_ @ w)
    w  = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints).x
    er = w @ mu_
    sd = np.sqrt(w @ cov_ @ w)
    y  = float(min(max((er - rf_) / (A * sd**2), 0), 1))
    return np.append(w * y, 1 - y)

TEST_START = "2020-01-01"
bt = []
for year in sorted(set(monthly.loc[TEST_START:].index.year)):
    train = monthly.loc[: f"{year-1}-12-31"]                 # past data only
    test  = monthly.loc[f"{year}-01-01": f"{year}-12-31"]    # the next year
    w = tangency_weights(train)
    bt.append(test[risky + [CASH_TICKER]] @ w)
bt = pd.concat(bt)

comp = pd.DataFrame({"Trustwise portfolio": bt,
                     "100% world equities": monthly.loc[bt.index, BENCHMARK],
                     "Cash": monthly.loc[bt.index, CASH_TICKER]})
growth = 100_000 * (1 + comp).cumprod()                     # value of £100k

summary_bt = pd.DataFrame({
    "return % / year": comp.mean() * 12 * 100,
    "volatility %":    comp.std() * np.sqrt(12) * 100,
    "Sharpe":          (comp.mean()*12 - comp["Cash"].mean()*12) / (comp.std()*np.sqrt(12)),
    "max drawdown %":  comp.apply(max_drawdown) * 100,
    "£100k becomes":   growth.iloc[-1],
})
display(summary_bt.round(2))

plt.figure(figsize=(9, 5))
for col, c in zip(growth.columns, [TEAL, NAVY, GREY]):
    plt.plot(growth.index, growth[col], color=c, lw=2.4 if col == "Trustwise portfolio" else 1.6, label=col)
    plt.annotate(f"£{growth[col].iloc[-1]:,.0f}", (growth.index[-1], growth[col].iloc[-1]), color=c, fontsize=9, fontweight="bold")
plt.title(f"£100k invested in {TEST_START[:4]}: out-of-sample test")
plt.ylabel("Value (£)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig("figures/fig3_backtest.png", dpi=200)
plt.show()

**Why this avoids overfitting:** few funds, simple weight limits, and weights chosen only with past data. We never tune the model on the test period.

# 9. K-means clustering: which charities are our customers? (Sessions 5–6)
Data: **Charity Commission for England and Wales — Register of Charities, public data extract** (open data, updated daily).
If the automatic download fails, download `publicextract.charity.zip` (JSON) from the Charity Commission "Register of Charities: data download" page, upload it to the Colab folder and run this section again.

In [ ]:
import requests

URL = "https://ccewuksprdoneregsadata1.blob.core.windows.net/data/json/publicextract.charity.zip"

if os.path.exists("publicextract.charity.zip"):
    data = open("publicextract.charity.zip", "rb").read()       # file uploaded by hand
else:
    data = requests.get(URL, timeout=120).content               # automatic download
z = zipfile.ZipFile(io.BytesIO(data))
with z.open(z.namelist()[0]) as f:
    reg = pd.read_json(io.TextIOWrapper(f, encoding="utf-8-sig"))

print("Rows in register:", len(reg))
print("Columns available:", list(reg.columns))

In [ ]:
# Cleaning: registered main charities (not linked subsidiaries), with income above £100k
df = reg.copy()
if "charity_registration_status" in df:
    df = df[df["charity_registration_status"].str.lower() == "registered"]
if "linked_charity_number" in df:
    df = df[df["linked_charity_number"] == 0]
df = df[["registered_charity_number", "charity_name", "latest_income", "latest_expenditure"]].dropna()
df = df[df["latest_income"] > 100_000]

# Two features: size (log of income) and spending ratio (spending / income)
df["log_income"]     = np.log10(df["latest_income"])
df["spending_ratio"] = (df["latest_expenditure"] / df["latest_income"]).clip(0, 3)
X = StandardScaler().fit_transform(df[["log_income", "spending_ratio"]])   # same scale for both features

# Choose k with the elbow method and the silhouette score (Session 6)
ks = range(2, 9)
sample = np.random.choice(len(X), min(20000, len(X)), replace=False)      # silhouette on a sample (faster)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X[sample], km.labels_[sample]))
best_k = list(ks)[int(np.argmax(sil))]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(ks, inertia, "o-", color=NAVY)
ax[0].set_title("Elbow method")
ax[0].set_xlabel("k")
ax[1].plot(ks, sil, "o-", color=TEAL)
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("k")
plt.tight_layout()
plt.show()
print("Number of clusters chosen (best silhouette):", best_k)

In [ ]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED).fit(X)
df["cluster"] = km.labels_

profile_tbl = df.groupby("cluster").agg(charities=("charity_name", "count"),
                                        median_income=("latest_income", "median"),
                                        median_spending_ratio=("spending_ratio", "median"))
# Target clusters: mid-sized (median income £500k-£10m) and spending less than they earn (they build up cash)
profile_tbl["target"] = profile_tbl["median_income"].between(5e5, 1e7) & (profile_tbl["median_spending_ratio"] < 1)
display(profile_tbl.round(2))

sam = df["latest_income"].between(5e5, 1e7).sum()
# Target = in a target cluster AND in our SAM income band, so TARGET is part of the SAM
target_clusters = profile_tbl.index[profile_tbl["target"]]
in_target = df["cluster"].isin(target_clusters) & df["latest_income"].between(5e5, 1e7)
target_n = int(in_target.sum())
print(f"Charities with income £500k-£10m (our SAM): {sam:,}")
print(f"Charities in our target cluster(s): {target_n:,}")

plt.figure(figsize=(9, 5))
plot = df.sample(min(15000, len(df)), random_state=RANDOM_SEED)
for c in sorted(plot["cluster"].unique()):
    s = plot[plot["cluster"] == c]
    is_target = profile_tbl.loc[c, "target"]
    plt.scatter(s["latest_income"], s["spending_ratio"], s=4, alpha=0.5,
                color=TEAL if is_target else GREY, label=f"Cluster {c}" + (" (target)" if is_target else ""))
plt.xscale("log")
plt.axhline(1, color=NAVY, lw=1, ls=":")
plt.xlabel("Yearly income (£, log scale)")
plt.ylabel("Spending ÷ income")
plt.title("Charities in England & Wales: our target groups in green")
plt.legend(markerscale=4, fontsize=8)
plt.tight_layout()
plt.savefig("figures/fig4_kmeans.png", dpi=200)
plt.show()

*Limitation:* the main register file has income and spending, but not cash balances. Charities that spend less than they earn are the ones that build up reserves, so we use this as a simple signal. With more time we would add the "annual return part B" file, which has balance-sheet data for larger charities.

# 10. Summary for the PDF
All the numbers used in the five-pager, in one place.

In [ ]:
best = summary_bt.loc["Trustwise portfolio"]
cash = summary_bt.loc["Cash"]
print("=== EXAMPLE CHARITY ===")
print(f"Reserves £{charity['total_reserves']:,.0f} | safe cash £{safe_cash:,.0f} | investable £{investable:,.0f} | profile {final_profile}")
print("\n=== PORTFOLIO ===")
display(weights)
print(f"Expected return {exp_1y:.1%} | GARCH volatility {vol_1y:.1%} | bad year (1 in 20) {bad_ret:+.1%} = £{bad_ret*investable:+,.0f}")
print("\n=== STRESS TESTS ===")
display(stress.round(1))
print("\n=== BACKTEST (out of sample) ===")
display(summary_bt.round(2))
print(f"\n=== MARKET === target charities: {target_n:,} | income £500k-£10m: {sam:,}")
print("\nCharts saved in figures/:", sorted(os.listdir("figures")))

---
**Sources:** Yahoo Finance (ETF prices); justETF and iShares / Amundi factsheets (fund details); Bank of England (Bank Rate); Charity Commission for England and Wales, Register of Charities public extract (open data).
**AI use:** we used an AI assistant to help draft and debug parts of this notebook; the team chose the methods, checked every result and ran the notebook.

# Part B — Business plan
## 11. Inputs (every number has a source or is a plan decision)
This is the Business Plan team's 36-month model (`Charity_Copilot_Financials_v2.xlsx`), rebuilt in Python so that the PDF comes straight from the notebook.
**Sourced inputs** come from public research (links on page 5). **Plan decisions** are what we choose to spend and when.

In [ ]:
INPUTS = dict(
    # Prices
    starter_month = 37,        # £/month = Beacon CRM Starter (leading UK charity CRM)
    board_year    = 1800,      # £/yr = top of a consultant investment-policy review (IIR £1,000-1,800 + VAT)
    partner_disc  = 0.20,      # reseller discount: low end of typical 20-30% (Monetizely; registered resellers 15-20%)
    # Market (Charity Commission, income bands, 23 May 2026)
    sam_bands     = (5935, 6349, 1228),   # £500k-1m, £1m-5m, £5m-10m
    tam           = 15209,                # all charities with income > £500k
    # SaaS benchmarks
    conv          = 0.182,     # trial-to-paid conversion (First Page Sage), used for pilots, firms, clients
    churn_year    = 0.1625,    # annual customer churn, median (Lighter Capital 2025)
    cac_ratio     = 1.63,      # £ sales & marketing per £1 of new ARR (Benchmarkit 2026)
    gross_margin  = 0.67,      # software firms < $1M revenue (Benchmarkit)
    # People (GOV.UK, ITJobsWatch, Payscale, HMRC)
    nlw = 12.71, hours = 37.5, dev_salary = 55000, cs_salary = 44664,
    ni = 0.15, ni_threshold = 5000, pension = 0.03, pension_lo = 6240, pension_hi = 50270,
    employment_allowance = 10500,
    # Other costs
    data_usd_month = 399, gbpusd = 1.34,          # EODHD market data; Reuters FX
    compliance = (3000 + 8000) / 2,               # Zitadelle 2026, midpoint
    fca_principal = (5000 + 25000) / 2,           # from Year 3, midpoint
    insurance = 532 + 750,                         # Simply Business + cyber (GetIndemnity)
    premoney = 2_000_000,                          # UK seed median pre-money (Beauhurst Q3 2026)
    # Plan decisions
    partner_reach = 1.0,       # share of SAM that partner firms reach (1.0 = base case; 0.5 tested on page 5)
    pilots = 10, paid_start = 7, partner_first = 7, partner_last = 30, founders = 2,
    developers = (2, 2, 2), cs_staff = (0, 1, 1), marketing = (15000, 20000, 40000),
)

## 12. 36-month model
Month by month: marketing creates direct customers (Starter/Board, split like the SAM income bands); accountancy firms are approached from month 7 to 30 and 18.2% sign; 18.2% of their clients adopt over the next 12 months; every customer group churns 16.25% a year.

In [ ]:
def full_cost(salary, I):
    '''Salary + employer National Insurance + employer pension (UK rules, GOV.UK).'''
    ni      = I["ni"] * max(0, salary - I["ni_threshold"])
    pension = I["pension"] * max(0, min(salary, I["pension_hi"]) - I["pension_lo"])
    return salary + ni + pension

def run_model(I, months=36):
    '''36-month plan, one row per month. I = the INPUTS dictionary.'''
    starter_price = I["starter_month"] * 12                     # £ per year
    partner_price = I["board_year"] * (1 - I["partner_disc"])   # Board plan with the partner discount
    sam = sum(I["sam_bands"])
    starter_share = I["sam_bands"][0] / sam                      # smallest charities choose Starter
    churn_m = 1 - (1 - I["churn_year"]) ** (1 / 12)              # yearly churn -> monthly churn

    founder_cost = full_cost(I["nlw"] * I["hours"] * 52, I)       # founders paid the National Living Wage
    dev_cost     = full_cost(I["dev_salary"], I)
    cs_cost      = full_cost(I["cs_salary"], I)
    other_costs  = I["data_usd_month"] * 12 / I["gbpusd"] + I["compliance"] + I["insurance"]   # per year

    rows = []
    starter, board, partner = 0.0, 0.0, 0.0   # number of paying customers of each type
    reached = []                              # charities reached through partner firms, month by month
    cum_cash = 0.0
    for m in range(1, months + 1):
        year = min((m - 1) // 12 + 1, 3)
        selling = m >= I["paid_start"]

        # 1) Direct sales: marketing + customer success spending creates new yearly revenue (CAC ratio)
        sales_spend = I["cs_staff"][year-1] * cs_cost / 12
        if selling:
            sales_spend = sales_spend + I["marketing"][year-1] / 12
        new_revenue = sales_spend / I["cac_ratio"] if selling else 0
        new_starter = new_revenue * starter_share / starter_price
        new_board   = new_revenue * (1 - starter_share) / I["board_year"]
        if m == I["paid_start"]:
            new_board = new_board + I["pilots"] * I["conv"]          # pilot charities that convert

        # 2) Partners: firms approached between partner_first and partner_last; their clients adopt over 12 months
        if I["partner_first"] <= m <= I["partner_last"]:
            reached.append(I["partner_reach"] * sam * I["conv"] / (I["partner_last"] - I["partner_first"] + 1))
        else:
            reached.append(0)
        new_partner = sum(reached[-12:]) * I["conv"] / 12

        # 3) Customers after churn
        starter = starter * (1 - churn_m) + new_starter
        board   = board * (1 - churn_m) + new_board
        partner = partner * (1 - churn_m) + new_partner

        # 4) Revenue, costs, profit
        revenue = (starter * starter_price + board * I["board_year"] + partner * partner_price) / 12
        staff   = (I["founders"] * founder_cost + I["developers"][year-1] * dev_cost) / 12
        fixed   = (other_costs + (I["fca_principal"] if year >= 3 else 0) - I["employment_allowance"]) / 12
        costs   = staff + sales_spend + fixed + (1 - I["gross_margin"]) * revenue
        cum_cash = cum_cash + revenue - costs
        rows.append(dict(month=m, year=year, starter=starter, board=board, partner=partner,
                         revenue=revenue, costs=costs, profit=revenue - costs, cum_cash=cum_cash))
    return pd.DataFrame(rows)

def key_outputs(df, I):
    peak = -df["cum_cash"].min()
    ask = int(np.ceil(peak * 1.2 / 50_000) * 50_000)
    prof = df.loc[df["profit"] > 0, "month"]
    return dict(first_profit_month=int(prof.min()) if len(prof) else None, peak_cash_need=peak, funding_ask=ask,
                equity=ask / (I["premoney"] + ask), y3_profit=df[df.year == 3]["profit"].sum())

model = run_model(INPUTS)
annual = model.groupby("year").agg(revenue=("revenue", "sum"), costs=("costs", "sum"), profit=("profit", "sum"))
ends = model[model.month.isin([12, 24, 36])].set_index("year")
annual["Starter (direct)"] = ends["starter"].round()
annual["Board (direct)"] = ends["board"].round()
annual["Via partners"] = ends["partner"].round()
annual["Paying charities"] = (ends["starter"] + ends["board"] + ends["partner"]).round()
annual["Cumulative cash"] = ends["cum_cash"]
annual.index = ["Year 1", "Year 2", "Year 3"]
K = key_outputs(model, INPUTS)
sam = sum(INPUTS["sam_bands"])
display(annual.style.format("{:,.0f}"))
print(f"First monthly profit: month {K['first_profit_month']} | Peak cash need: £{K['peak_cash_need']:,.0f}")
print(f"Funding ask: £{K['funding_ask']:,.0f} for {K['equity']:.1%} equity (pre-money £{INPUTS['premoney']:,.0f})")
print(f"Paying charities end of Year 3: {annual.loc['Year 3','Paying charities']:,.0f} = {annual.loc['Year 3','Paying charities']/sam:.1%} of SAM")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(model["month"], model["revenue"], color=TEAL, label="Monthly revenue")
ax.bar(model["month"], -model["costs"], color=NAVY, alpha=0.85, label="Monthly costs")
ax2 = ax.twinx()
ax2.plot(model["month"], model["cum_cash"], color=ORANGE, lw=2.2, label="Cumulative cash")
ax2.axhline(0, color=GREY, lw=0.8)
if K["first_profit_month"]:
    ax.axvline(K["first_profit_month"], color=ORANGE, ls=":", lw=1.2)
    ax.text(K["first_profit_month"] + 0.3, ax.get_ylim()[1] * 0.85, f"break-even\nmonth {K['first_profit_month']}", color=ORANGE, fontsize=8)
for a in (ax, ax2):
    a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"£{v/1000:,.0f}k"))
ax.set_xlabel("Month"); ax.set_title("Trustwise: 36-month plan (base case)")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig("figures/fig5_business_plan.png", dpi=200); plt.show()

## 13. Unit economics and sensitivity: what has to be true

In [ ]:
I = INPUTS
churn = I["churn_year"]; gm = I["gross_margin"]
unit = pd.DataFrame({
    "Board (direct)":  [I["board_year"], I["board_year"] * gm / churn, I["board_year"] * I["cac_ratio"]],
    "Starter (direct)": [I["starter_month"] * 12, I["starter_month"] * 12 * gm / churn, I["starter_month"] * 12 * I["cac_ratio"]],
    "Partner client":  [I["board_year"] * (1 - I["partner_disc"]), I["board_year"] * (1 - I["partner_disc"]) * gm / churn, 0],
}, index=["Annual price (£)", "Lifetime gross profit (£)", "Cost to acquire (£)"])
unit.loc["LTV : CAC"] = [unit.iloc[1, i] / unit.iloc[2, i] if unit.iloc[2, i] else np.nan for i in range(3)]
display(unit.round(1))
print(f"Customer lifetime = 1 / churn = {1/churn:.1f} years. SaaS median LTV:CAC ~3.0x (KeyBanc 2025): direct sales alone are below it, so partners lead.")

cases = [("Base", {}), ("Conversion 20%", {"conv": 0.20}), ("Conversion 22%", {"conv": 0.22}),
         ("Conversion 25%", {"conv": 0.25}), ("Conversion 35% (top quartile)", {"conv": 0.35}),
         ("Churn 10%", {"churn_year": 0.10}), ("Churn 25%", {"churn_year": 0.25}),
         ("Sales cost £1.14 per £1", {"cac_ratio": 1.14}), ("Sales cost £2.22 per £1", {"cac_ratio": 2.22})]
sens = {}
for name, change in cases:
    k = key_outputs(run_model({**INPUTS, **change}), INPUTS)
    sens[name] = {"First profitable month": k["first_profit_month"] or "none", "Year-3 profit (£)": round(k["y3_profit"]),
                  "Peak cash need (£)": round(k["peak_cash_need"])}
sens = pd.DataFrame(sens).T
display(sens)

# Part C — Build the five-pager PDF
The next cell writes the page layout (HTML + CSS). **To change any text in the PDF, edit it in that cell.**
Words in double curly brackets, like `{{TAKEAWAY}}`, are filled in automatically with the results above.

In [ ]:
# Cover photo, stored inside the notebook (no upload needed). Free-licence photo to be cited in the sources.
import base64
COVER_B64 = '...'
open('cover_crop.jpg', 'wb').write(base64.b64decode(COVER_B64))
print('Cover photo ready: cover_crop.jpg')

In [ ]:
# Screenshots of the live demo website (page 2), stored inside the notebook.
DEMO_B64 = '...'
open('demo_shot.jpg', 'wb').write(base64.b64decode(DEMO_B64))
print('Demo screenshot ready: demo_shot.jpg')

In [ ]:
%%writefile template.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Charity Investment Copilot – Five-Pager (draft)</title>
<style>
@import url("https://fonts.googleapis.com/css2?family=Montserrat:wght@600;700;800;900&family=Inter:wght@400;600;700;800&display=swap");
@page { size: A4; margin: 0; }
:root { --navy:#14213D; --teal:#167A68; --teal2:#E4F2EE; --orange:#D9682B; --paper:#FFFFFF; --soft:#F6F4EF; --line:#D9D3C7; --muted:#5B6476; }
* { box-sizing: border-box; }
body { margin:0; font-family: Inter, Arial, sans-serif; color: var(--navy); font-size: 8.6pt; line-height: 1.38; }
.page { width:210mm; height:297mm; padding:17mm 12mm 10mm; background: url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 210 30' preserveAspectRatio='none'><path d='M0 0 H210 V18 Q120 2 0 9 Z' fill='%23167A68'/><path d='M120 0 H210 V8 Q170 1 120 0 Z' fill='%2314213D'/></svg>") no-repeat top left; background-size:210mm 26mm; page-break-after: always; position: relative; overflow: hidden; }
.page:last-child { page-break-after: auto; }

/* header */
.kicker { font-size:6.8pt; letter-spacing:.14em; font-weight:700; color: var(--teal); text-transform: uppercase; }
.kicker span { color: var(--muted); }
h1 { font-family: Montserrat, "Inter Display", Inter, sans-serif; font-size:20pt; font-weight:800; margin:1mm 0 1.5mm; letter-spacing:-.01em; text-transform: uppercase; }
.rule { height:0.8mm; width:32mm; background: var(--teal); margin-bottom:4mm; }

/* boxes */
.sec { display:flex; align-items:center; gap:2.5mm; margin:3.5mm 0 2mm; }
.sec .ic { width:6.5mm; height:6.5mm; border-radius:50%; background: var(--teal); color:#fff; font-weight:800; font-size:8pt; text-align:center; line-height:6.5mm; }
.sec h2 { font-family:Montserrat, Inter; font-size:11pt; margin:0; font-weight:800; color: var(--teal); }
.box { border:0.3mm solid var(--line); border-radius:2.5mm; padding:3mm 3.5mm; background: var(--paper); }
.box.soft { background: var(--soft); border-color: var(--soft); }
.box.teal { background: var(--teal2); border-color: var(--teal2); }
.box.warn { border-left:1.2mm solid var(--orange); }
.box h3 { font-size:9pt; margin:0 0 1mm; font-weight:800; }
.box p { margin:0 0 1.5mm; }
.box p:last-child { margin-bottom:0; }
.grid2 { display:flex; gap:3mm; }
.grid2 > * { flex:1; }
.grid3 { display:flex; gap:3mm; }
.grid3 > * { flex:1; }
.grid4 { display:flex; gap:2.5mm; }
.grid4 > * { flex:1; }
.w60 { flex:1.6 !important; }
ul { margin:1mm 0 0; padding-left:4mm; }
li { margin-bottom:0.8mm; }
.small { font-size:7.2pt; color: var(--muted); }
.big { font-family:Montserrat, "Inter Display", Inter; font-size:20pt; font-weight:800; color: var(--teal); line-height:1; margin-bottom:1mm; }
.step { font-family:Montserrat, "Inter Display", Inter; font-size:15pt; font-weight:800; color: var(--orange); line-height:1; margin-bottom:1mm; }
.ph { border:0.5mm dashed var(--orange); border-radius:2.5mm; color: var(--orange); font-weight:700; text-align:center; padding:4mm; background:#FDF5EF; }
.todo { color: var(--orange); font-weight:700; }

table { width:100%; border-collapse: collapse; font-size:7.8pt; }
th { background: var(--navy); color:#fff; text-align:left; padding:1.6mm 2mm; font-weight:700; }
td { padding:1.4mm 2mm; border-bottom:0.25mm solid var(--line); vertical-align: top; }
tr:nth-child(even) td { background: var(--soft); }
td.c, th.c { text-align:center; }

.btn { display:inline-block; background: var(--orange); color:#fff; font-weight:800; padding:2mm 5mm; border-radius:10mm; text-decoration:none; font-size:9pt; }
.foot { position:absolute; bottom:5mm; left:12mm; right:12mm; font-size:6.8pt; color: var(--muted); display:flex; justify-content:space-between; border-top:0.25mm solid var(--line); padding-top:1.5mm; }
.band { background: var(--navy); color:#fff; border-radius:2.5mm; padding:3mm 4mm; }
.band b { color:#8FD9C6; }
.compact td { padding:0.9mm 2mm; } .compact th { padding:1.1mm 2mm; }
.phase th { background: var(--teal); text-align:center; }
.phase td { text-align:center; }

/* cover */
.cover .cta { position:absolute; top:9mm; right:16mm; background:#14213D; color:#fff; text-decoration:none;
  border-radius:10mm; padding:1.6mm 1.6mm 1.6mm 5mm; font-family:Montserrat, Inter; font-weight:800; font-size:9.5pt; }
.cover .cta-txt { display:inline-block; vertical-align:middle; margin-right:3mm; }
.cover .cta-arrow { display:inline-block; vertical-align:middle; width:7mm; height:7mm; line-height:7mm; text-align:center;
  border-radius:50%; background:#167A68; color:#fff; font-size:10pt; }

.cover { padding:0; background:#F5F7F6; }
.cover .photo { position:absolute; left:0; top:28mm; width:210mm; height:170mm; object-fit:cover; object-position:50% 0%; }
.cover svg.shapes { position:absolute; top:0; left:0; width:210mm; height:297mm; }
.cover .brand { position:absolute; top:9mm; left:16mm; }
.cover .brand .logo { width:16mm; height:16mm; display:inline-block; vertical-align:middle; }
.cover .brand .wm { display:inline-block; vertical-align:middle; margin-left:3mm; }
.cover .brand .name { font-family:Montserrat, Inter; font-weight:900; font-size:23pt; letter-spacing:.03em; line-height:1; color:#14213D; }
.cover .brand .name span { color:#167A68; }
.cover .brand .sub { font-family:Montserrat, Inter; font-weight:700; font-size:7.6pt; letter-spacing:.16em; color:#5B6476; margin-top:1.2mm; }
.cover .ctitle { position:absolute; top:203mm; left:16mm; width:120mm; font-family:Montserrat, Inter; font-weight:900; font-size:38pt; line-height:1.0; color:#14213D; }
.cover .ctitle span { color:#167A68; }
.cover .ctag { position:absolute; top:240mm; left:16mm; width:105mm; font-size:10.5pt; line-height:1.4; color:#14213D; border-left:1.2mm solid #167A68; padding-left:3mm; }
.cover .cstats { position:absolute; top:205mm; right:16mm; width:58mm; }
.cover .cstat { background:#fff; border-radius:2.5mm; padding:2.2mm 3.5mm; margin-bottom:2.2mm; border-left:1.2mm solid #167A68; }
.cover .cstat b { display:block; font-family:Montserrat, Inter; font-weight:900; font-size:15pt; color:#167A68; line-height:1.05; }
.cover .cstat span { font-size:7.5pt; color:#5B6476; }
.cover .cbar { position:absolute; left:0; right:0; bottom:0; height:16mm; background:#14213D; color:#C9D3E3; font-size:8pt;
  display:flex; justify-content:space-between; align-items:center; padding:0 16mm; }
.cover .cbar b { color:#8FD9C6; letter-spacing:.12em; margin-right:2mm; }

/* pricing cards */
.price { border:0.3mm solid var(--line); border-radius:2.5mm; padding:3mm 3.5mm; background:#fff; position:relative; }
.price .pname { font-family:Montserrat, Inter; font-weight:800; font-size:9pt; color:var(--teal); text-transform:uppercase; letter-spacing:.08em; }
.price .pval { font-family:Montserrat, Inter; font-weight:900; font-size:19pt; line-height:1.1; margin:0.8mm 0 1.2mm; }
.price .pval span { font-size:8pt; font-weight:600; color:var(--muted); }
.price p { margin:0; font-size:8pt; }
.price .pnote { margin-top:1.2mm; font-size:7pt; color:var(--muted); font-style:italic; }
.price.hot { background:var(--navy); color:#fff; border-color:var(--navy); }
.price.hot .pname { color:#8FD9C6; } .price.hot .pval span, .price.hot .pnote { color:#C9D3E3; }
.price .pbadge { position:absolute; top:3mm; right:3mm; background:var(--teal); color:#fff; font-size:6.5pt; font-weight:700; padding:0.6mm 2mm; border-radius:5mm; }
/* chevron timeline */
.chev { display:flex; gap:1.5mm; }
.chev > div { flex:1; background:var(--teal2); padding:2.2mm 3mm 2.2mm 5mm; font-size:7.8pt; line-height:1.3; border-left:1.2mm solid var(--teal); border-radius:0 2mm 2mm 0; }
.chev > div b { display:block; color:var(--teal); font-family:Montserrat, Inter; font-size:8.3pt; margin-bottom:0.5mm; }
/* risk cards */
.risk { border:0.3mm solid var(--line); border-top:1.2mm solid #E0A800; border-radius:2.5mm; padding:2.5mm 3mm; background:#fff; }
.risk.hi { border-top-color:var(--orange); } .risk.lo { border-top-color:var(--teal); }
.risk .rlev { font-size:6.5pt; font-weight:800; letter-spacing:.12em; color:var(--muted); }
.risk h3 { font-size:9pt; margin:0.5mm 0 1mm; } .risk p { margin:0; font-size:8pt; }
.tagline { font-size:6.5pt; font-weight:800; letter-spacing:.12em; color:var(--teal); }

/* timeline */
.tl { position:relative; display:flex; gap:3mm; padding-top:1mm; }
.tl:before { content:""; position:absolute; left:6mm; right:6mm; top:5.2mm; height:0.6mm; background:linear-gradient(90deg,#A8DCCD,#14213D); }
.tl-step { flex:1; position:relative; }
.tl-dot { width:8.5mm; height:8.5mm; border-radius:50%; color:#fff; font-family:Montserrat, Inter; font-weight:800; font-size:9pt;
  line-height:8.5mm; text-align:center; position:relative; border:0.8mm solid #fff; }
.tl-step:nth-child(1) .tl-dot { background:#3FB497; } .tl-step:nth-child(2) .tl-dot { background:#167A68; }
.tl-step:nth-child(3) .tl-dot { background:#1F4A5E; } .tl-step:nth-child(4) .tl-dot { background:#14213D; }
.tl-step b { display:block; font-family:Montserrat, Inter; font-size:8.6pt; margin-top:1.5mm; color:#14213D; }
.tl-step .when { display:inline-block; font-size:6.8pt; font-weight:700; color:#167A68; letter-spacing:.06em; text-transform:uppercase; margin:0.4mm 0 0.8mm; }
.tl-step p { margin:0; font-size:7.8pt; line-height:1.3; color:#5B6476; }

.pill { position:absolute; top:16mm; right:12mm; background:#14213D; color:#fff; text-decoration:none; border-radius:10mm;
  padding:1.3mm 1.3mm 1.3mm 4mm; font-family:Montserrat, Inter; font-weight:800; font-size:8pt; }
.pill .cta-txt { display:inline-block; vertical-align:middle; margin-right:2.5mm; }
.pill .cta-arrow { display:inline-block; vertical-align:middle; width:5.8mm; height:5.8mm; line-height:5.8mm; text-align:center;
  border-radius:50%; background:#167A68; color:#fff; font-size:8.5pt; }
</style>
</head>
<body>

<!-- ===================== COVER ===================== -->
<div class="page cover">
  <img class="photo" src="cover_crop.jpg" alt="Charity board meeting">
  <svg class="shapes" viewBox="0 0 210 297" preserveAspectRatio="none" xmlns="http://www.w3.org/2000/svg">
    <!-- white area above the photo -->
    <path d="M0 0 H210 V20 Q112 70 0 46 Z" fill="#F5F7F6"/>
    <!-- top teal curve -->
    <path d="M0 40 Q110 66 210 16 V32 Q118 78 0 52 Z" fill="#167A68"/>
    <!-- white area below the photo -->
    <path d="M0 190 Q110 160 210 184 V297 H0 Z" fill="#F5F7F6"/>
    <!-- bottom navy curve -->
    <path d="M0 176 Q110 146 210 170 V186 Q110 162 0 192 Z" fill="#14213D"/>
  </svg>
  <div class="brand">
    <svg class="logo" viewBox="0 0 40 40" xmlns="http://www.w3.org/2000/svg" aria-label="Trustwise logo">
      <circle cx="20" cy="20" r="19" fill="#167A68"/>
      <path d="M20 31 V17" stroke="#fff" stroke-width="2.4" stroke-linecap="round"/>
      <path d="M20 21 C20 14 24.5 10 30 10 C30 16 25.5 20 20 21 Z" fill="#fff"/>
      <path d="M20 25 C20 20 16.5 17 12 17 C12 21.5 15.5 25 20 25 Z" fill="#fff" opacity="0.75"/>
    </svg>
    <div class="wm">
      <div class="name">TRUST<span>WISE</span></div>
      <div class="sub">CHARITY INVESTMENT COPILOT</div>
    </div>
  </div>

  <a class="cta" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
  <div class="ctitle">BUSINESS<br>PLAN <span>2026</span></div>
  <div class="ctag">Helping charities put their idle cash to work: safely, cheaply and in line with the rules.</div>
  <div class="cstats">
    <div class="cstat"><b>£31bn</b><span>held in cash by larger UK charities</span></div>
    <div class="cstat"><b>26%</b><span>of charities did not review their reserves policy last year</span></div>
    <div class="cstat"><b>3.5%</b><span>UK inflation, Sept 2026</span></div>
  </div>
  <div class="cbar"><span><b>DATE</b> 15 November 2026</span><span>KX09 · Data Science for Asset Management</span><span>Anonymous group submission</span></div>

</div>

<!-- ===================== PAGE 1 ===================== -->
<div class="page">
<a class="pill" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
<div class="kicker">Trustwise <span>· Executive summary</span></div>
<h1>The Problem &amp; Our Answer</h1>
<div class="rule"></div>

<div class="sec"><div class="ic">1</div><h2>The Idea in One Minute</h2></div>
<div class="grid2">
  <div class="box w60">
  <p>Every charity keeps money aside for bad times: its <b>reserves</b>. Many leave all of it in a bank account for years, because the people in charge, the <b>trustees</b>, are volunteers afraid of investing it badly. But cash loses value when prices rise, and the regulator now expects trustees to explain and manage this money.</p>
  <p><b>Trustwise</b> is a simple online tool for trustees. In four steps it tells a charity <b>how much it can safely invest</b>, <b>where</b> (a few low-cost funds), <b>what a bad year could cost in pounds</b>, and it writes the <b>report the board needs</b>.</p>
  </div>
  <div class="box teal"><h3>Who it is for</h3>
  <p>Mid-sized charities (income £500k–£10m) that hold reserves in cash but cannot pay a wealth manager or a consultant every year.</p>
  <p style="margin-top:1.5mm"><b>How it is sold:</b> a flat yearly fee, directly and through charities' accountants.</p></div>
</div>

<div class="sec"><div class="ic">2</div><h2>The Problem in Numbers</h2></div>
<div class="grid3">
  <div class="box"><div class="big">£31bn</div><p>held in cash by the 12,973 charities with income above £0.5m in 2022. That is 13% of their £250bn of assets. <span class="small">(Broadstone, 2024, on 2022 accounts)</span></p></div>
  <div class="box"><div class="big">58%</div><p>of these charities (7,551) hold <b>no long-term investments</b> at all (2022). <span class="small">(Broadstone, 2024)</span></p></div>
  <div class="box"><div class="big">£71k</div><p>is what £100k earning nothing is worth after 10 years, if prices rise 3.5% a year. <span class="small">(our calculation)</span></p></div>
</div>
<div class="grid2" style="margin-top:3mm">
  <div class="box soft"><h3>Why trustees don't invest</h3>
  <ul><li>They fear making a mistake or losing money</li><li>Most are volunteers: only 47% of charities rate their trustees' investment knowledge as good <span class="small">(Brewin Dolphin)</span></li><li>Advice is expensive: wealth managers charge about 0.75% a year, consultants £1,000–£1,800 per review</li></ul></div>
  <div class="box soft"><h3>Why now</h3>
  <ul><li><b>16 Sept 2026:</b> new reserves guidance (CC19) no longer accepts "boilerplate" policies</li><li><b>Nov 2026:</b> Bank Rate up to 4%, inflation at 3.5%</li><li>About <b>4 in 10</b> charities with £500k–£5m income spent more than they earned in 2024</li></ul></div>
</div>

<div class="sec"><div class="ic">3</div><h2>What the Rules Say</h2></div>
<table class="compact">
<tr><th style="width:27%">Rule</th><th>What it means for trustees</th></tr>
<tr><td>Trustee Act 2000, s.3–5</td><td>Trustees may invest. They must check the investment is suitable, spread the risk, review it and take proper advice.</td></tr>
<tr><td>CC14 Investment guidance (2023)</td><td>Keeping spare cash on deposit <b>is itself an investment decision</b>, with the same duties.</td></tr>
<tr><td>CC19 Reserves guidance (Sept 2026)</td><td>Charities must explain why they hold their reserves and how much. High, unexplained reserves attract questions.</td></tr>
</table>

<div class="sec"><div class="ic">4</div><h2>Our Answer: Four Simple Steps</h2></div>
<div class="grid4">
  <div class="box"><div class="step">01</div><h3>Safe amount</h3><p>We keep enough cash for emergencies and planned spending. Only the rest can be invested.</p></div>
  <div class="box"><div class="step">02</div><h3>Simple portfolio</h3><p>A mix of a few low-cost funds, matched to how much risk the charity can take.</p></div>
  <div class="box"><div class="step">03</div><h3>Bad year in £</h3><p>Before investing, trustees see what a bad year could cost, in pounds.</p></div>
  <div class="box"><div class="step">04</div><h3>Board report</h3><p>A ready report that shows the board followed CC14 and CC19.</p></div>
</div>
<div class="band" style="margin-top:3mm"><p style="margin:0">Why it matters: every <b>extra 1% a year</b> earned on the sector's £31bn of cash means about <b>£310m more each year</b> for charitable work.</p></div>
<div class="foot"><span>Sources: full list on page 5</span><span>1</span></div>
</div>

<!-- ===================== PAGE 2 ===================== -->
<div class="page">
<a class="pill" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
<div class="kicker">Trustwise <span>· Product &amp; technology</span></div>
<h1>How It Works</h1>
<div class="rule"></div>

<div class="sec"><div class="ic">1</div><h2>Product Modules and the Course Methods Behind Them</h2></div>
<table>
<tr><th style="width:21%">Module</th><th>What it does for the trustee</th><th style="width:33%">Method</th><th class="c" style="width:8%">Session</th></tr>
<tr><td><b>1. Reserve Check</b></td><td>Keeps 3–12 months of spending in cash (more if income is unstable) and ring-fences restricted funds. Shows the amount that can be invested.</td><td>Data cleaning, EDA; risk capacity vs tolerance, final profile = the lower of the two</td><td class="c">2, 4</td></tr>
<tr><td><b>2. Portfolio Builder</b></td><td>Builds a mix of 8 low-cost ETFs plus a cash fund, with limits (max 40% per fund, gold ≤ 10%).</td><td>Efficient frontier, tangency portfolio, CAL; y* = (E[R<sub>p</sub>] − R<sub>f</sub>) / (A·σ²), capped 0–1</td><td class="c">4</td></tr>
<tr><td><b>3. Bad-Year Lens</b></td><td>Shows a 1-in-20 bad year in pounds and replays 2008, 2020 and 2022.</td><td>GARCH(1,1) volatility forecast, 95% VaR and CVaR; historical stress tests</td><td class="c">3, 8</td></tr>
<tr><td><b>4. Board Report</b></td><td>Writes a plain-English report for the board, aligned with CC14 and CC19.</td><td>Out-of-sample backtest with yearly rebalancing to avoid overfitting</td><td class="c">9</td></tr>
<tr><td><b>5. Sector Map</b></td><td>Places the charity among similar charities from the public register.</td><td>K-means clustering, chosen by elbow method and silhouette score</td><td class="c">5–6</td></tr>
</table>

<div class="sec"><div class="ic">2</div><h2>What Makes Us Different</h2></div>
<div class="grid4">
  <div class="box"><h3>Independent</h3><p>We do not sell our own funds, so we have no reason to push one.</p></div>
  <div class="box"><h3>Risk in pounds</h3><p>"{{BADYEAR_EX}}" is clearer than "volatility 10%".</p></div>
  <div class="box"><h3>Rules built in</h3><p>The output is the evidence the board needs for CC14 and CC19.</p></div>
  <div class="box"><h3>Flat price</h3><p>From £444 a year, not a % of the money invested.</p></div>
</div>

<div class="sec"><div class="ic">3</div><h2>Data &amp; Tools</h2></div>
<div class="grid2">
  <div class="box"><h3>Data</h3><ul>
  <li><b>ETF prices:</b> daily adjusted prices (dividends included) from Yahoo Finance for 9 London-listed GBP funds + benchmark, Jun 2014 – Nov 2026. We fix a known pence/pound error in the feed.</li>
  <li><b>Stress tests:</b> long-history index ETFs as proxies for 2008.</li>
  <li><b>Charities:</b> Charity Commission register (open data, ~171k charities): income and spending.</li>
  <li><b>Economy:</b> Bank of England Bank Rate, ONS inflation.</li></ul></div>
  <div class="box"><h3>Tools</h3><ul>
  <li><b>Python</b> in one Jupyter notebook: analysis → business plan → this PDF.</li>
  <li>pandas, NumPy; SciPy (SLSQP optimiser); arch (GARCH); scikit-learn (K-means, silhouette); Matplotlib.</li>
  <li>WeasyPrint to build this document from the notebook.</li>
  <li>React + TypeScript website (live demo): questionnaire → board summary, same numbers as this notebook.</li></ul></div>
</div>

<div class="sec"><div class="ic">4</div><h2>The Demo</h2></div>
<a href="#" style="display:block"><img src="demo_shot.jpg" style="width:100%; display:block" alt="Trustwise demo website"></a>
<div style="display:flex; justify-content:space-between; font-size:7pt; color:#5B6476; margin-top:1.2mm">
<span><b style="color:#167A68">1 · Home</b> — the problem and how it works</span>
<span><b style="color:#167A68">2 · Reserves check</b> — 16 plain-English questions</span>
<span><b style="color:#167A68">3 · Board summary</b> — what to invest, bad year, board report</span></div>
<div class="foot"><span>Code: <a href="https://github.com/andap2303-tech/trustwise" style="color:inherit">github.com/andap2303-tech/trustwise</a></span><span>2</span></div>
</div>

<!-- ===================== PAGE 3 ===================== -->
<div class="page">
<a class="pill" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
<div class="kicker">Trustwise <span>· Analysis &amp; results</span></div>
<h1>Proof That It Works</h1>
<div class="rule"></div>

<div class="sec"><div class="ic">1</div><h2>Our Three Model Portfolios: Where the Money Goes</h2></div>
{{ALLOC_TABLE}}
<p class="small" style="margin-top:0.8mm">Built from low-cost, London-listed index funds in GBP (average cost ~0.1% a year). Weights = tangency portfolio × y*, the rest in a cash fund (Session 4); max 40% per fund, gold ≤ 10%.{{BOND_NOTE}} <b>Why so much Japan, and no UK shares?</b> The optimiser looks only at 2014–2026 risk and return, and our fund list has no UK equity fund; Japan scored well on return per unit of risk. Weights show the method, not a recommendation. Equity funds are unhedged (USD, EUR, JPY risk). Full fund list in the notebook.</p>

<div class="sec"><div class="ic">2</div><h2>Key Charts</h2></div>
<div class="grid2">
  <img src="figures/fig1_frontier.png" style="width:100%; height:43mm; object-fit:contain; border:0.3mm solid #D9D3C7; border-radius:2.5mm; background:#fff">
  <img src="figures/fig2_garch_bad_year.png" style="width:100%; height:43mm; object-fit:contain; border:0.3mm solid #D9D3C7; border-radius:2.5mm; background:#fff">
</div>
<div class="grid2" style="margin-top:3mm">
  <img src="figures/fig3_backtest.png" style="width:100%; height:39mm; object-fit:contain; border:0.3mm solid #D9D3C7; border-radius:2.5mm; background:#fff">
  <img src="figures/fig4_kmeans.png" style="width:100%; height:39mm; object-fit:contain; border:0.3mm solid #D9D3C7; border-radius:2.5mm; background:#fff">
</div>

<div class="sec"><div class="ic">3</div><h2>Stress Tests</h2></div>
<table class="compact">
<tr><th>Scenario</th><th class="c">Our portfolio</th><th class="c">100% equities</th><th class="c">Cash</th></tr>
{{STRESS_ROWS}}
</table>
<div class="box teal" style="margin-top:3mm"><h3>Key takeaway</h3><p>{{TAKEAWAY}}</p></div>
<div class="foot"><span>All numbers from our notebook</span><span>3</span></div>
</div>

<!-- ===================== PAGE 4 ===================== -->
<div class="page">
<a class="pill" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
<div class="kicker">Trustwise <span>· Market &amp; business model</span></div>
<h1>Market &amp; Business Model</h1>
<div class="rule"></div>

<div class="grid2">
<div>
<div class="sec" style="margin-top:0"><div class="ic">1</div><h2>Market Size</h2></div>
<svg viewBox="0 0 100 74" style="width:100%; height:58mm" xmlns="http://www.w3.org/2000/svg" aria-label="TAM SAM SOM">
  <circle cx="37" cy="37" r="36" fill="#DCEFEA"/>
  <circle cx="37" cy="42" r="31" fill="#A8DCCD"/>
  <circle cx="37" cy="52" r="21" fill="#3FB497"/>
  <circle cx="37" cy="63" r="10" fill="#14213D"/>
  <g font-family="Montserrat, Inter" font-weight="800" font-size="4.4" text-anchor="middle">
    <text x="37" y="7.6" fill="#14213D">TAM</text>
    <text x="37" y="22.6" fill="#14213D">SAM</text>
    <text x="37" y="43.6" fill="#fff">TARGET</text>
    <text x="37" y="64.6" fill="#fff">SOM</text>
  </g>
  <g font-family="Inter" font-size="2.9" fill="#5B6476">
    <line x1="49" y1="6.2" x2="76" y2="6.2" stroke="#5B6476" stroke-width="0.25"/>
    <text x="77" y="5.4"><tspan font-weight="700" fill="#14213D">15,209</tspan></text><text x="77" y="8.8">&gt;£500k · May 2026</text>
    <line x1="49" y1="21.2" x2="76" y2="21.2" stroke="#5B6476" stroke-width="0.25"/>
    <text x="77" y="20.4"><tspan font-weight="700" fill="#14213D">13,512</tspan></text><text x="77" y="23.8">£500k–£10m · {{SAM_VALUE}}</text>
    <line x1="52" y1="42.2" x2="76" y2="42.2" stroke="#5B6476" stroke-width="0.25"/>
    <text x="77" y="41.4"><tspan font-weight="700" fill="#14213D">{{TARGET_N}}</tspan></text><text x="77" y="44.8">K-means cluster</text>
    <line x1="45" y1="63.2" x2="76" y2="63.2" stroke="#5B6476" stroke-width="0.25"/>
    <text x="77" y="62.4"><tspan font-weight="700" fill="#14213D">{{SOM}}</tspan></text><text x="77" y="65.8">Year 3 · {{SOM_PCT}} of SAM</text>
  </g>
</svg>
<p class="small">171,474 charities in England &amp; Wales (Charity Commission, May 2026). Every charity above £40k income has an accountant: our sales channel.</p>
</div>
<div>
<div class="sec" style="margin-top:0"><div class="ic">2</div><h2>Where We Fit</h2></div>
<svg viewBox="0 0 100 74" style="width:100%; height:58mm" xmlns="http://www.w3.org/2000/svg" aria-label="Positioning map">
  <rect x="10" y="2" width="88" height="62" fill="#F6F4EF" rx="2"/>
  <rect x="10" y="2" width="44" height="31" fill="#E4F2EE" rx="2"/>
  <line x1="54" y1="2" x2="54" y2="64" stroke="#D9D3C7" stroke-width="0.3"/>
  <line x1="10" y1="33" x2="98" y2="33" stroke="#D9D3C7" stroke-width="0.3"/>
  <text x="12" y="31" font-family="Montserrat, Inter" font-weight="800" font-size="2.8" fill="#167A68">SWEET SPOT</text>
  <g font-family="Inter" font-size="2.8" fill="#5B6476">
    <text x="54" y="70" text-anchor="middle">Cost for a mid-sized charity  →  higher</text>
    <text x="5" y="33" text-anchor="middle" transform="rotate(-90 5 33)">Built for trustee decisions →</text>
  </g>
  <g font-family="Inter" font-size="3" fill="#14213D">
    <circle cx="80" cy="14" r="2.2" fill="#5B6476"/><text x="76" y="9.5" text-anchor="middle">Consultant</text><text x="76" y="20.5" text-anchor="middle" font-size="2.5" fill="#5B6476">£1–1.8k/review</text>
    <circle cx="88" cy="28" r="2.2" fill="#5B6476"/><text x="88" y="38" text-anchor="middle">Wealth</text><text x="88" y="41" text-anchor="middle">manager</text><text x="88" y="44.2" text-anchor="middle" font-size="2.5" fill="#5B6476">~0.75%/yr</text>
    <circle cx="68" cy="50" r="2.2" fill="#5B6476"/><text x="68" y="56" text-anchor="middle">Pooled fund</text><text x="68" y="59" text-anchor="middle" font-size="2.5" fill="#5B6476">~0.85%/yr</text>
    <circle cx="26" cy="56" r="2.2" fill="#5B6476"/><text x="26" y="51.5" text-anchor="middle">Robo-advisor</text><text x="26" y="62" text-anchor="middle" font-size="2.5" fill="#5B6476">individuals only</text>
    <circle cx="26" cy="15" r="4" fill="#167A68"/><text x="26" y="8.5" text-anchor="middle" font-weight="800" font-family="Montserrat, Inter">Trustwise</text><text x="26" y="23.5" text-anchor="middle" font-size="2.5" fill="#167A68">£444–£1,800 flat</text>
  </g>
</svg>
<p class="small">On £500k, a pooled fund costs ~£4,250 a year. Trustwise Board costs £1,800 and can sit alongside any fund. (CCLA, IIR, Rathbones, Nutmeg)</p>
</div>
</div>

<div class="sec"><div class="ic">3</div><h2>Pricing: A Flat Yearly Fee, Not a % of the Money</h2></div>
<div class="grid3">
  <div class="price"><div class="pname">Starter</div><div class="pval">£444<span>/year</span></div><p><b>For:</b> smaller charities.<br><b>Gets:</b> cash buffer, model portfolio, bad year in £.</p><p class="pnote">Same price as a charity CRM (£37/month)</p></div>
  <div class="price hot"><div class="pbadge">Core plan</div><div class="pname">Board</div><div class="pval">£1,800<span>/year</span></div><p><b>For:</b> boards that must decide.<br><b>Gets:</b> Starter + CC14/CC19 report, stress tests, yearly review.</p><p class="pnote">Costs the same as one consultant review</p></div>
  <div class="price"><div class="pname">Partner</div><div class="pval">£1,440<span>/client/yr</span></div><p><b>For:</b> accountants with charity clients.<br><b>Gets:</b> Board plan for every client, in one view.</p><p class="pnote">Board plan with a 20% partner discount</p></div>
</div>

<p class="small" style="margin-top:1.2mm"><b style="color:#167A68">Coming in Year 3: Board+ Custom.</b> Portfolios built for each charity, via an FCA partner. Upside, not in our numbers.</p>

<div class="sec"><div class="ic">4</div><h2>Go-to-Market: Partners First</h2></div>
<div class="tl">
  <div class="tl-step"><div class="tl-dot">1</div><b>Pilot</b><span class="when">Months 1–6</span><p>10 charities test for free, via one accountant</p></div>
  <div class="tl-step"><div class="tl-dot">2</div><b>Partners</b><span class="when">Months 7–30</span><p>Accountancy firms offer Trustwise to their clients</p></div>
  <div class="tl-step"><div class="tl-dot">3</div><b>Scale</b><span class="when">Year 2</span><p>First sales hire; target the K&#8209;means cluster</p></div>
  <div class="tl-step"><div class="tl-dot">4</div><b>Regulated</b><span class="when">Year 3</span><p>FCA partner firm; personal advice</p></div>
</div>

<div class="sec"><div class="ic">5</div><h2>Financial Outlook (base case, median SaaS benchmarks)</h2></div>
{{FIN_TABLE}}
<div class="foot"><span>Base case: 18.2% conversion, 16.25% churn, £1.63 sales cost per £1 of new revenue, 67% gross margin. Sources on page 5.</span><span>4</span></div>
</div>

<!-- ===================== PAGE 5 ===================== -->
<div class="page">
<a class="pill" href="#"><span class="cta-txt">Try our live demo</span><span class="cta-arrow">→</span></a>
<div class="kicker">Trustwise <span>· Rules, risks &amp; vision</span></div>
<h1>Rules, Risks &amp; Next Steps</h1>
<div class="rule"></div>

<div class="sec"><div class="ic">1</div><h2>The Rules We Work Within</h2></div>
<div class="grid2">
  <div class="box"><h3>Different money, different rules</h3>
  <p><span class="tagline">UNRESTRICTED</span> Free to use: invested only above the safe cash level.</p>
  <p><span class="tagline">RESTRICTED</span> For a specific project: kept in cash or short bonds until spent.</p>
  <p><span class="tagline">ENDOWMENT</span> Capital must be kept: long-term, total-return portfolio.</p></div>
  <div class="box warn"><h3>Where we stand with the FCA</h3>
  <p><b>Phase 1 (Years 1–2) · Decision support.</b> Generic model portfolios, education and a board report. Trustees decide. Flat fee, not a % of assets.</p>
  <p><b>Phase 2 (Year 3+) · Regulated advice.</b> Personal advice through an FCA-authorised principal firm (£5k–£25k a year, already in our costs).</p></div>
</div>

<div class="sec"><div class="ic">2</div><h2>Key Risks &amp; How We Handle Them</h2></div>
<div class="grid3">
  <div class="risk hi"><div class="rlev">HIGH</div><h3>Partner conversion</h3><p>{{PARTNER_RISK}}</p></div>
  <div class="risk hi"><div class="rlev">HIGH</div><h3>Cash runway</h3><p>Profit only from month {{BE_MONTH}}. {{ASK}} raise covers it; hire only as revenue grows.</p></div>
  <div class="risk"><div class="rlev">MEDIUM</div><h3>Advice regulation</h3><p>Generic portfolios in phase 1; FCA partner in year 3.</p></div>
</div>
<div class="grid3" style="margin-top:2.5mm">
  <div class="risk"><div class="rlev">MEDIUM</div><h3>"Cash is fine now"</h3><p>At 4% Bank Rate: lead with the CC14/CC19 duty and the report, not returns.</p></div>
  <div class="risk"><div class="rlev">MEDIUM</div><h3>Strong incumbents</h3><p>CCLA, Broadstone: be the decision layer, not a fund; partner with them.</p></div>
  <div class="risk lo"><div class="rlev">LOW</div><h3>Model risk</h3><p>GARCH and history can miss extreme events: limits stated in every report.</p></div>
</div>

<div class="sec"><div class="ic">3</div><h2>Limits of Our Analysis (and What We Would Do Next)</h2></div>
<div class="box soft"><ul style="margin:0">
<li><b>Short history:</b> data from June 2014; the 2020–2026 test was a strong bull market, where our risk-adjusted return (Sharpe) was slightly below world equities. Our aim is smaller falls, not beating equities.</li>
<li><b>Stress tests:</b> 2008 uses US-listed index ETFs in USD, so currency effects are approximate.</li>
<li><b>Normal VaR:</b> prudent (no expected return counted), but a 2008-type crash is still worse. <i>Next:</i> Student-t GARCH.</li>
<li><b>K-means proxy:</b> the register has no cash data, so "spends less than it earns" stands in for "holds idle cash".</li>
<li><b>Business model:</b> median SaaS benchmarks, not real customers; the pilot will measure conversion and churn.</li>
</ul></div>

<div class="sec"><div class="ic">4</div><h2>Impact &amp; Long-Term Vision</h2></div>
<p style="margin:0 0 1mm">Better returns on reserves fund more charitable work: <b>+1% a year on £31bn ≈ £310m a year</b>, and consultant-level analysis becomes affordable from £444 a year.</p>
<div class="tl" style="margin-top:1mm">
  <div class="tl-step"><div class="tl-dot">1</div><b>The standard tool</b><span class="when">Years 1–3</span><p>The CC14/CC19 tool used by charity accountants</p></div>
  <div class="tl-step"><div class="tl-dot">2</div><b>Regulated advice</b><span class="when">Years 3–5</span><p>Custom ethical portfolios via an FCA partner; Scotland's charities</p></div>
  <div class="tl-step"><div class="tl-dot">3</div><b>Open benchmark</b><span class="when">Year 5+</span><p>Public data on charity cash; ESG fund screening</p></div>
</div>

<div class="sec"><div class="ic">5</div><h2>Sources &amp; Links</h2></div>
<p class="small" style="line-height:1.4"><b>Charities:</b> Charity Commission, income bands (May 2026), Annual Return 2024, register extract, Research with Trustees 2026 · Broadstone via Financial Planning Today and CFG (2024) · Rathbones, State of the Sector (Nov 2026) · RSM trustee survey (Dec 2024) · Brewin Dolphin. <b>Rules:</b> Trustee Act 2000 · CC14 (2023) · CC19 and accounting thresholds (Browne Jacobson, Autumn 2026). <b>Economy:</b> Bank of England MPR (Nov 2026) · ONS CPI (Sept 2026). <b>Competitors &amp; prices:</b> CCLA, Cazenove, Rathbones, IIR, Nutmeg, Beacon CRM. <b>Benchmarks:</b> First Page Sage, Lighter Capital, Benchmarkit, KeyBanc, Beauhurst Q3 2026, Carta. <b>Costs:</b> GOV.UK, ITJobsWatch, Payscale, Xero, EODHD, Zitadelle, Simply Business. <b>Funds:</b> justETF, Yahoo Finance. <b>Course:</b> KX09 Sessions 1–9.
<b>AI use:</b> an AI assistant helped find sources, draft text and write code; the team checked every figure and ran the notebook.</p>
<div class="foot"><span>Anonymous submission · KX09 · Code: <a href="https://github.com/andap2303-tech/trustwise" style="color:#167A68; font-weight:700">github.com/andap2303-tech/trustwise</a></span><span>5</span></div>
</div>

</body>
</html>

In [ ]:
from weasyprint import HTML

html = open("template.html", encoding="utf-8").read()
DEMO_URL = "https://trustwise1.netlify.app/"   # live demo website
html = html.replace('href="#"', f'href="{DEMO_URL}"')     # every "Try our demo / Try it yourself" button
money = lambda v: ("-" if v < 0 else "") + f"£{abs(v):,.0f}"
k_ = lambda v: ("-" if v < 0 else "") + f"£{abs(v)/1000:,.0f}k"

# Page 3: stress tests and key takeaway
rows = ""
for name, r in stress.iterrows():
    rows += (f"<tr><td>{name}</td><td class='c'>{r['Our portfolio (%)']:+.1f}%</td>"
             f"<td class='c'>{r['100% world equities (%)']:+.1f}%</td><td class='c'>{r['Cash (%)']:+.1f}%</td></tr>\n")
best, cash, eq = summary_bt.loc["Trustwise portfolio"], summary_bt.loc["Cash"], summary_bt.loc["100% world equities"]
falls = stress[(stress["100% world equities (%)"] < 0) & (stress["Our portfolio (%)"] < 0)]
half = (falls["Our portfolio (%)"] / falls["100% world equities (%)"]).mean() if len(falls) else np.nan
if 0.4 <= half <= 0.6:
    crash_txt = "lost <b>about half as much as world equities</b>"
elif 0 < half < 1:
    crash_txt = f"lost <b>about {half:.0%} of what world equities lost</b>"
else:
    crash_txt = "behaved as shown in the stress-test table"
takeaway = (f"Out of sample, £100k invested in {TEST_START[:4]} grew to <b>£{best['£100k becomes']:,.0f}</b> "
            f"(cash £{cash['£100k becomes']:,.0f}; world equities £{eq['£100k becomes']:,.0f}). "
            f"In 2008, 2020 and 2022 the Balanced portfolio {crash_txt}. "
            f"Example charity, £{investable:,.0f} invested: a 1-in-20 bad year costs <b>{money(abs(bad_ret*investable))}</b>, "
            f"a 2008-type crash <b>{money(abs(stress.iloc[0]['Our change in £']))}</b>. That is why the cash buffer comes first.")

# Page 3: allocation table for the three profiles
TER = {"CSP1.L": 0.07, "SMEA.L": 0.12, "SJPA.L": 0.12, "CPJ1.L": 0.20, "EMIM.L": 0.18, "SGLN.L": 0.12,
       "IGLT.L": 0.07, "SLXX.L": 0.20, CASH_TICKER: 0.09}            # % a year, justETF 13 Nov 2026
NAMES = {"low": "Careful", "medium": "Balanced", "high": "Growth"}
# Group funds into regions / asset classes (no fund names in the PDF)
GROUPS = [("US equities", ["CSP1.L"], "#14213D"), ("Europe equities", ["SMEA.L"], "#1F4A5E"),
          ("Japan equities", ["SJPA.L"], "#167A68"), ("Asia-Pacific & EM equities", ["CPJ1.L", "EMIM.L"], "#3FB497"),
          ("Gold", ["SGLN.L"], "#C9A227"), ("UK bonds", ["IGLT.L", "SLXX.L"], "#8E9AAF"), ("Cash fund", [CASH_TICKER], "#D9E2EC")]
def group_weights(p):
    y = y_star(A_BY_PROFILE[p]); w = dict(zip(risky, w_tan * y)); w[CASH_TICKER] = 1 - y
    return [(name, sum(w[t] for t in ts), col) for name, ts, col in GROUPS]
alloc = "<div style='margin-bottom:1.5mm'>"
for p in LEVELS:
    gw = group_weights(p)
    eq = sum(v for n, v, c in gw if "equities" in n)
    segs = "".join(f"<div style='width:{v*100:.2f}%; background:{c}; color:{'#fff' if c not in ('#D9E2EC', '#C9A227') else '#14213D'}; "
                   f"text-align:center; font-size:7pt; font-weight:700; line-height:7mm; overflow:hidden'>{v*100:.0f}%</div>"
                   if v >= 0.045 else (f"<div style='width:{v*100:.2f}%; background:{c}'></div>" if v > 0.002 else "") for n, v, c in gw)
    alloc += (f"<div style='display:flex; align-items:center; margin-bottom:1.6mm'>"
              f"<div style='width:27mm; font-weight:800; font-family:Montserrat, Inter; font-size:8.5pt'>{NAMES[p]}{' ★' if p == final_profile else ''}"
              f"<div style='font-weight:400; font-family:Inter; font-size:6.8pt; color:#5B6476'>{eq:.0%} in equities</div></div>"
              f"<div style='flex:1; display:flex; height:7mm; border-radius:1.5mm; overflow:hidden'>{segs}</div></div>")
alloc += "<div style='display:flex; flex-wrap:wrap; gap:3mm; margin:0.5mm 0 0 27mm; font-size:6.8pt; color:#5B6476'>" + "".join(
    f"<span><span style='display:inline-block; width:2.5mm; height:2.5mm; background:{c}; border-radius:0.5mm; margin-right:1mm'></span>{n}</span>"
    for n, ts, c in GROUPS) + "</div></div>"
def prof_stats(p):
    y = y_star(A_BY_PROFILE[p]); w = np.append(w_tan * y, 1 - y)
    er, sd = rf + y * (er_t - rf), y * vol_risky              # same GARCH risk as the bad-year chart
    return er, sd, -1.645 * sd, sum(wi * TER[t] for wi, t in zip(w, risky + [CASH_TICKER]))
st = {p: prof_stats(p) for p in LEVELS}
alloc += "<table class='compact'><tr><th></th>" + "".join(f"<th class='c'>{NAMES[p]}{' ★' if p == final_profile else ''}</th>" for p in LEVELS) + "</tr>"
for k, lab, fmt in [(0, "Expected return / year (2014–26 data)", "{:.1%}"), (1, "Volatility / year (GARCH)", "{:.1%}"),
                    (2, "Bad year (1 in 20, prudent)", "{:+.1%}"), (3, "Fund costs / year", "{:.2f}%")]:
    alloc += f"<tr><td>{lab}</td>" + "".join(f"<td class='c'><b>{fmt.format(st[p][k])}</b></td>" for p in LEVELS) + "</tr>"
alloc += "</table>"
bond_w = w_tan[risky.index("IGLT.L")] + w_tan[risky.index("SLXX.L")]
bond_note = (" UK bonds get ~0%: in 2014–2026 they gave low returns with high risk (2022 crash), "
             "so the optimiser uses the cash fund as the safe part." if bond_w < 0.01 else "")

# Page 4: financial table + key boxes
Y = annual
fin = "<div class='grid2'><div class='w60'><table class='compact'><tr><th></th><th class='c'>Year 1</th><th class='c'>Year 2</th><th class='c'>Year 3</th></tr>"
fin += "<tr><td>Paying charities (end of year)</td>" + "".join(f"<td class='c'>{v:,.0f}</td>" for v in Y["Paying charities"]) + "</tr>"
fin += "<tr><td>&nbsp;&nbsp;of which via partner accountants</td>" + "".join(f"<td class='c'>{v:,.0f}</td>" for v in Y["Via partners"]) + "</tr>"
for col, lab in [("revenue", "Revenue"), ("costs", "Costs"), ("profit", "<b>Profit / loss</b>")]:
    fin += f"<tr><td>{lab}</td>" + "".join(f"<td class='c'>{k_(v)}</td>" for v in Y[col]) + "</tr>"
fin += "</table></div><img src='figures/fig5_business_plan.png' style='width:100%; height:33mm; object-fit:contain'></div>"
s20 = sens.loc["Conversion 20%"]
fin += (f"<div class='grid3' style='margin-top:2mm'>"
        f"<div class='box'><h3>Break-even</h3><p>First monthly profit in <b>month {K['first_profit_month']}</b>. "
        f"At 20% conversion: month {s20['First profitable month']}, Year 3 {k_(s20['Year-3 profit (£)'])}.</p></div>"
        f"<div class='box'><h3>Funding ask</h3><p><b>£{K['funding_ask']/1000:,.0f}k for {K['equity']:.0%} equity</b> "
        f"(UK seed median pre-money £2.0m). Covers the peak cash need of {k_(K['peak_cash_need'])}.</p></div>"
        f"<div class='box'><h3>Why partners</h3><p>A direct Board client: £{unit.loc['Lifetime gross profit (£)','Board (direct)']/1000:,.1f}k lifetime gross profit "
        f"vs £{unit.loc['Cost to acquire (£)','Board (direct)']/1000:,.1f}k to win (LTV:CAC {unit.loc['LTV : CAC','Board (direct)']:.1f}x). "
        f"Partners sell for us: no sales spend, 20% off.</p></div></div>"
        )

som = annual.loc["Year 3", "Paying charities"]
sens_line = f"20% conversion brings break-even to month {s20['First profitable month']}"
try:
    target_txt = f"{target_n:,}"
except NameError:
    target_txt = "[run section 9]"
half = run_model(dict(INPUTS, partner_reach=0.5)); Kh = key_outputs(half, INPUTS)
partner_risk = (f"{annual.loc['Year 3','Via partners']:,.0f} of {som:,.0f} Year-3 clients come via partners. At half the reach: "
                + (f"break-even only in month {Kh['first_profit_month']}" if Kh['first_profit_month'] else "no break-even by month 36")
                + f", peak cash £{Kh['peak_cash_need']/1000:,.0f}k. The pilot tests reach first.")
sam_value = INPUTS["sam_bands"][0] * INPUTS["starter_month"] * 12 + sum(INPUTS["sam_bands"][1:]) * INPUTS["board_year"]   # smallest band on Starter, others on Board
bad_ex = f"A bad year could cost {money(abs(bad_ret*investable))}" if bad_ret < 0 else f"Even a bad year stays at {money(bad_ret*investable)}"

for key, val in {"{{STRESS_ROWS}}": rows, "{{ALLOC_TABLE}}": alloc, "{{BOND_NOTE}}": bond_note, "{{TAKEAWAY}}": takeaway, "{{FIN_TABLE}}": fin, "{{TARGET_N}}": target_txt,
                 "{{SOM}}": f"{som:,.0f}", "{{SOM_PCT}}": f"{som/sum(INPUTS['sam_bands']):.1%}",
                 "{{SENS_LINE}}": sens_line, "{{BE_MONTH}}": str(K["first_profit_month"]),
                 "{{ASK}}": f"£{K['funding_ask']/1000:,.0f}k", "{{BADYEAR_EX}}": bad_ex, "{{PARTNER_RISK}}": partner_risk, "{{SAM_VALUE}}": f"£{sam_value/1e6:,.0f}m/yr"}.items():
    html = html.replace(key, val)
open("five_pager.html", "w", encoding="utf-8").write(html)
HTML("five_pager.html", base_url=".").write_pdf("Trustwise_Five_Pager.pdf")
print("PDF created: Trustwise_Five_Pager.pdf  (download it from the folder on the left)")

In [ ]:
# Preview page 4 (Proof that it works) inside the notebook
!pip install -q pdf2image && apt-get -qq install -y poppler-utils > /dev/null
from pdf2image import convert_from_path
convert_from_path("Trustwise_Five_Pager.pdf", dpi=70)[3]